In [2]:
import numpy as np
import pandas as pd
import requests
import math
from scipy import stats
from scipy.stats import percentileofscore as score
import xlsxwriter
from datetime import date, timedelta
import json


In [3]:
stocks = pd.read_csv("sp_500_stocks.csv")

In [4]:
api_key= "UTyHVduohyBpg6IFqApjemQi7MtN1dTB"

In [9]:
symbol = "^GSPC"
today = date.today()
one_year_ago = today - timedelta(days=((365*10)))
api_url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{symbol}?from={one_year_ago}&to={today}&apikey={api_key}"
data = requests.get(api_url).json()

In [10]:
sp500 = pd.DataFrame(data['historical'])

sp500 

# 3. Process the DataFrame for better usability (optional but recommended)
sp500['date'] = pd.to_datetime(sp500['date']) # Convert 'date' column to datetime objects
sp500.set_index('date', inplace=True)      # Set the 'date' column as the index
sp500 = sp500.iloc[::-1]      # Reverses the order
del sp500['adjClose']
del sp500['unadjustedVolume']
del sp500['label']
sp500


,open,high,low,close,volume,change,changePercent,vwap,changeOverTime
date,,,,,,,,,
2015-08-17,2089.70,2102.87,2079.30,2102.44,2867690000,12.74,0.609660,2094.8700,0.006097
2015-08-18,2101.99,2103.47,2094.14,2096.92,2949990000,-5.07,-0.241200,2098.1800,-0.002412
2015-08-19,2095.69,2096.17,2070.53,2079.61,3512920000,-16.08,-0.767290,2082.1000,-0.007673
2015-08-20,2076.61,2076.61,2035.73,2035.73,3922470000,-40.88,-1.970000,2049.3600,-0.019700
2015-08-21,2034.08,2034.08,1970.89,1970.89,5018240000,-63.19,-3.110000,1991.9500,-0.031100
...,...,...,...,...,...,...,...,...,...
2025-08-07,6379.43,6389.71,6310.32,6339.99,5306090000,-39.44,-0.618240,6354.8625,-0.006182
2025-08-08,6355.22,6395.16,6355.22,6389.44,4769910000,34.22,0.538460,6373.7600,0.005385
2025-08-11,6389.67,6407.25,6364.06,6373.46,4652400000,-16.21,-0.253690,6383.6100,-0.002537


In [11]:
sp500["Tomorrow"] = sp500["close"].shift(-1)
sp500

,open,high,low,close,volume,change,changePercent,vwap,changeOverTime,Tomorrow
date,,,,,,,,,,
2015-08-17,2089.70,2102.87,2079.30,2102.44,2867690000,12.74,0.609660,2094.8700,0.006097,2096.92
2015-08-18,2101.99,2103.47,2094.14,2096.92,2949990000,-5.07,-0.241200,2098.1800,-0.002412,2079.61
2015-08-19,2095.69,2096.17,2070.53,2079.61,3512920000,-16.08,-0.767290,2082.1000,-0.007673,2035.73
2015-08-20,2076.61,2076.61,2035.73,2035.73,3922470000,-40.88,-1.970000,2049.3600,-0.019700,1970.89
2015-08-21,2034.08,2034.08,1970.89,1970.89,5018240000,-63.19,-3.110000,1991.9500,-0.031100,1893.21
...,...,...,...,...,...,...,...,...,...,...
2025-08-07,6379.43,6389.71,6310.32,6339.99,5306090000,-39.44,-0.618240,6354.8625,-0.006182,6389.44
2025-08-08,6355.22,6395.16,6355.22,6389.44,4769910000,34.22,0.538460,6373.7600,0.005385,6373.46
2025-08-11,6389.67,6407.25,6364.06,6373.46,4652400000,-16.21,-0.253690,6383.6100,-0.002537,6445.75


In [12]:
sp500["Target"] = (sp500["Tomorrow"] > sp500["close"]).astype(int)
sp500

,open,high,low,close,volume,change,changePercent,vwap,changeOverTime,Tomorrow,Target
date,,,,,,,,,,,
2015-08-17,2089.70,2102.87,2079.30,2102.44,2867690000,12.74,0.609660,2094.8700,0.006097,2096.92,0
2015-08-18,2101.99,2103.47,2094.14,2096.92,2949990000,-5.07,-0.241200,2098.1800,-0.002412,2079.61,0
2015-08-19,2095.69,2096.17,2070.53,2079.61,3512920000,-16.08,-0.767290,2082.1000,-0.007673,2035.73,0
2015-08-20,2076.61,2076.61,2035.73,2035.73,3922470000,-40.88,-1.970000,2049.3600,-0.019700,1970.89,0
2015-08-21,2034.08,2034.08,1970.89,1970.89,5018240000,-63.19,-3.110000,1991.9500,-0.031100,1893.21,0
...,...,...,...,...,...,...,...,...,...,...,...
2025-08-07,6379.43,6389.71,6310.32,6339.99,5306090000,-39.44,-0.618240,6354.8625,-0.006182,6389.44,1
2025-08-08,6355.22,6395.16,6355.22,6389.44,4769910000,34.22,0.538460,6373.7600,0.005385,6373.46,0
2025-08-11,6389.67,6407.25,6364.06,6373.46,4652400000,-16.21,-0.253690,6383.6100,-0.002537,6445.75,1


In [13]:
len(sp500)

2513

In [35]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, class_weight="balanced", min_samples_split=2, random_state=67) 
# 1:4 ratio
trainsize = math.floor(len(sp500) / 5) 
#trainsize = 100
print(trainsize)
train = sp500.iloc[:trainsize]
test = sp500.iloc[(-4*trainsize):]
# can't overlap to prevent cross-validation
predictors = ["close", "volume", "open", "high", "low", "changePercent","vwap"]
model.fit(train[predictors], train["Target"])

502


RandomForestClassifier(class_weight='balanced', random_state=67)

In [36]:
from sklearn.metrics import precision_score

preds = model.predict(test[predictors])
preds = pd.Series(preds, index=test.index)
precision_score(test["Target"], preds)

0.5882352941176471

In [37]:
def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])
    preds = model.predict(test[predictors])
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined

In [41]:
def backtest(data, model, predictors, start=2500, step=100):
    all_predictions = []

    for i in range(start, data.shape[0], step):
        train = data.iloc[0:i].copy()
        test = data.iloc[i:(i+step)].copy()
        predictions = predict(train, test, predictors, model)
        all_predictions.append(predictions)
    
    return pd.concat(all_predictions)


In [42]:
predictions = backtest(sp500, model, predictors)

In [43]:
precision_score(predictions["Target"], predictions["Predictions"])

0.4166666666666667

In [27]:
datas

[{'symbol': 'SPY',
  'publishedDate': '2025-07-31 09:34:23',
  'title': 'NASDAQ Index, S&P 500 and Dow Jones Forecasts – US Indices Continue to Look Upwards',
  'image': 'https://images.financialmodelingprep.com/news/nasdaq-index-sp-500-and-dow-jones-forecasts-us-20250731.jpg',
  'site': 'fxempire.com',
  'text': 'The three major US indices all look as if they are going to continue to look to higher levels, although there are differing signs of momentum or strength.',
  'url': 'https://www.fxempire.com/forecasts/article/nasdaq-index-sp-500-and-dow-jones-forecasts-us-indices-continue-to-look-upwards-1537769'},
 {'symbol': 'SPY',
  'publishedDate': '2025-07-31 08:24:15',
  'title': 'How To Trade SPY, Top Tech Stocks Using Technical Analysis',
  'image': 'https://images.financialmodelingprep.com/news/how-to-trade-spy-top-tech-stocks-using-technical-20250731.jpg',
  'site': 'benzinga.com',
  'text': 'Good Morning Traders!',
  'url': 'https://www.benzinga.com/markets/equities/25/07/46754137

In [29]:
if datas and isinstance(datas, list): # Check if we got a list of data
    # This is the key step: convert the list of dictionaries to a DataFrame
    df = pd.DataFrame(datas)

    # 3. Best Practices: Clean & Refine the DataFrame
    # ===============================================
    
    # Convert the 'publishedDate' column to a proper datetime format
    df['publishedDate'] = pd.to_datetime(df['publishedDate'])

    # Set the 'publishedDate' as the index for easier time-series analysis
    df.set_index('publishedDate', inplace=True)
    
    # Sort by date, just in case the API doesn't return it sorted
    df.sort_index(ascending=False, inplace=True)

    # Optionally, select only the columns you need
    df_clean = df[['symbol', 'title', 'site', 'url']]

df_clean


,symbol,title,site,url
publishedDate,,,,
2025-07-31 09:34:23,SPY,"NASDAQ Index, S&P 500 and Dow Jones Forecasts ...",fxempire.com,https://www.fxempire.com/forecasts/article/nas...
2025-07-31 08:24:15,SPY,"How To Trade SPY, Top Tech Stocks Using Techni...",benzinga.com,https://www.benzinga.com/markets/equities/25/0...
2025-07-31 08:00:04,SPY,Can the S&P 500 Rally Overcome Bearish Seasona...,schaeffersresearch.com,https://www.schaeffersresearch.com/content/ana...
2025-07-31 04:20:00,SPY,Stock Market Today: S&P 500 Futures Rise After...,wsj.com,https://www.wsj.com/livecoverage/stock-market-...
2025-07-31 03:50:00,SPY,Stock Market Today: Flurry of Trade Deals Boos...,wsj.com,https://www.wsj.com/livecoverage/stock-market-...
2025-07-30 16:14:52,SPY,YLDE: A Dividend ETF That Writes Options Again...,seekingalpha.com,https://seekingalpha.com/article/4806339-ylde-...
2025-07-30 15:19:08,SPY,Wall Street bull boosts his S&P 500 target to ...,youtube.com,https://www.youtube.com/watch?v=-4X8qMBN9-M
2025-07-30 14:36:00,SPY,UPS Stock Joins 7% Yielders in S&P 500. Who El...,barrons.com,https://www.barrons.com/articles/ups-stock-yie...
2025-07-30 14:25:00,SPY,Is another S&P 500 change on the horizon? Here...,marketwatch.com,https://www.marketwatch.com/story/is-another-s...
